# Why does the trained decomposition of a broad fine-tune fail the adversarial check?

*E39, `experiments/broad_fragility`, 2026-10-27. Record: `reports/analyses/2026-10-27_analysis_broad-fragility.md`.
Plan: `reports/plans/2026-10-27_plan_broad-fragility.md`. Follows E37 and its Amendment 1,
`reports/analyses/2026-10-26_analysis_teacher-spread.md`. Figures from `experiments/broad_fragility/plot.py`.*

**Words.** **The decomposition after teacher training**, "the trained decomposition" for short, is the released
decomposition with its alive subcomponents trained to imitate a fine-tuned model. Γ and Δ stay frozen. **The check**
is the paper's adversarial check: a 20-step mask adversary from one random start on 256 Pile rows. It is read as the
state's KL over the unchanged decomposition's KL, so 1 means no more fragile than the original. It passes at or under
1.40 and fails at or over 1.61. **The excess** is how far the check sits above 1.

## The answer

The failure is not general brittleness of the decomposition: a random move of the same size per matrix passed. It
sits in the changes teacher training makes to the MLP subcomponents, and it needs most of those changes together.
Putting back any half of each matrix's subcomponents removed nearly all of it. The rarely important half removed
less of it than a random half, not more. A rank-8 LoRA fine-tune of the same task gave a trained decomposition that passes. SGD gave no answer,
because none of its rates learned.

> **Something unexpected happened that I cannot explain.** Four filed predictions went the other way. A random move
> of the trained decomposition's size passed. The MLP changes alone carried all the excess, and the attention changes
> hid part of it. The rarely important subcomponents mattered least. The LoRA's trained decomposition passed. Each is
> set out under Surprises.

## What we were trying to test

E37 trained the subcomponents to imitate a fine-tune that teaches the 67M model to write lowercase text, a change at
every position. The trained decomposition kept the whole gain but read 2.20 under the check. Training four times
longer did not bring it down. The user asked for cheap tests of why, and whether SGD or LoRA updates do better. All
three tests start from E37's trained decomposition, retrained inside the job by the same recipe.

- **Random moves.** Each matrix's alive $U$ and $V$ get Gaussian noise, scaled so each matrix's weight change has the
  size of the trained decomposition's change there. Two noise seeds at full size, one at half size. If the check
  punishes any move of this size, these fail too, and the failure is brittleness.
- **Reverts.** Put the released $U$ and $V$ back on a group of subcomponents and keep the trained values on the rest.
  The groups are each layer, all attention matrices, and all MLP matrices. Then, per matrix, halves of the alive
  subcomponents ranked by Γ's mean causal importance on the check's rows. The **rarely important half** tests one
  guess. Teacher training may have written part of the change into subcomponents Γ hardly ever switches on, which
  the adversary can then switch off cheaply. The **often important half** is the rest, and a **random half** is the
  null. A revert is scored as the **share of the excess removed**, $(r_\text{trained} - r_\text{revert})/(r_\text{trained} - 1)$
  with $r$ the check. 1 removes it all, 0 none, and a negative share makes it worse.
- **Other update shapes.** The same fine-tune made two more ways, both on all 24 matrices: plain SGD, and a rank-8
  LoRA trained with Adam. Each takes the rate, out of three, whose held-out gain is nearest the Adam fine-tune's.
  Each update is then placed three ways: **do-nothing**, which leaves it in Δ; the **scaled adjoint split**; and
  **teacher training** by E37's recipe. **Stable rank**, $\|\delta W\|_F^2/\|\delta W\|_2^2$ averaged over matrices,
  says how many directions an update spreads over.

Two more numbers are read at every state. **Behaviour kept** is the share of the fine-tune's held-out gain in mean
log-probability that the state keeps: 0 is the unchanged model, 1 the fine-tune. **Damage** is
$\mathrm{KL}(W_0 \,\|\, \text{state})$ on plain Pile text, in nats per token.

**Controls, run first, with an early stop.** All passed. The unchanged model gave the card's known seed spread of
0.0748 and E37's readings to four digits. The retrained decomposition read 2.2011, E37's value, and kept 0.998 of the
behaviour. Putting every subcomponent back read exactly as the unchanged model.

## What we expected, and why

- **A random move of the same size fails**, at 1.8 to 3. E37's fixes all ended near 2 whatever the route, which looked
  like a property of moving this far.
- **The half-size move reads lower**, a sanity check on the move.
- **The excess is spread over layers**, so no layer's revert removes more than 0.6 of it.
- **The rarely important half removes at least 0.2 more than the random half**, if the frozen Γ is the weak point.
- **The SGD and LoRA updates are less spread than Adam's by stable rank.** This is the third test's premise.
- **Their trained decompositions still fail.** If the failure belongs to broad behaviour, update shape should not
  matter.
- **Their do-nothing states fail**, since the whole update sits in Δ, which the adversary can switch off.

## Does a random move of the same size fail the check?

![random moves](../experiments/broad_fragility/figures/moves.png)

**Reading it.** No. Both full-size moves read 1.03 and 1.07, inside the grey band where the unchanged model sits
across adversary seeds. The half-size move reads 0.99. The trained decomposition, moved by the same amount in every
matrix, reads 2.20. So the check does not punish a move of the subcomponents as such. The right panel is the caveat.
The random moves barely change what the model predicts on plain text: their damage is about a twenty-fifth of the
trained decomposition's. Equal size in the weights is not equal size in the outputs. This rules out "any weight move
of this size fails", not "any move that changes the outputs this much fails". Low damage alone does not guarantee a
pass either. In E37's last amendment, a masked recipe with nothing to imitate failed at 3.14 with damage of 0.007.

## Which subcomponents carry the excess?

![reverts](../experiments/broad_fragility/figures/reverts.png)

**Reading it.** The MLP subcomponents' changes carry it, and only when nearly all of them are present. Putting back
every MLP matrix removes all of the excess, reading 1.00. Putting back every attention matrix instead raises the check
from 2.20 to 2.70, so the attention changes were offsetting part of it. By layer, layer 0 removes 0.70 of the excess
and layer 3 only 0.11. The four layer shares add to 1.44, so the effects do not add up. Any half of each matrix's
subcomponents removes nearly all of it: the random half 0.97, the often important half 0.96. The rarely important half
removes the least, 0.63, the opposite of the guess it tested.

The right panel shows the price in behaviour. Putting back the MLP matrices keeps less than half the behaviour. The
random half keeps all of it by mean log-probability, and passes. That is not a fix yet. Its KL to the fine-tuned model
rises from 0.012 to 0.075 nats, where the unchanged model reads 0.272. One caveat holds for the whole figure. Γ's
importance was read at the unchanged model, and even the often important half averages only 0.028. Both halves are off
at most positions, so the ranking may not separate what the adversary can switch off.

## Does a less spread-out update give a less fragile trained decomposition?

![update shape](../experiments/broad_fragility/figures/shape.png)

**Reading it.** For the LoRA, yes. Teacher training on the LoRA reads 1.27, under the pass line, keeping 0.96 of its
behaviour. On the Adam fine-tune the same recipe reads 2.20. The LoRA's update is far less spread, with a stable rank
of 1.9 against 28.5. Its outside share is a little lower, 0.26 against 0.32, and both sit at about half the random
level. Do-nothing fails for both, as expected. The adjoint split passes for both while keeping less behaviour.

Two caveats bear on reading the pass as an effect of rank. The LoRA's gain is 14% above the Adam fine-tune's, because
no rate in its grid came closer. A larger gain to imitate should make passing harder, so this mismatch does not
explain the pass. The second caveat might. The LoRA fine-tune changes plain text about half as much, damage 0.121
against 0.235. Its do-nothing state also reads lower, 2.65 against 4.87. So a smaller change away from the task may be
what passes, rather than the low rank; that was not tested. This is one rate and one seed.

**SGD gave no state.** Every rate in the grid made held-out lowercase text less likely, by 0.14 to 3.6 nats per token.
By the filed rule, its arm was not run. An amendment with smaller rates is planned.

In [1]:
# The reads behind the three figures, from the results file (no model).
import json
from pathlib import Path

r = json.loads(Path("../experiments/broad_fragility/results/broad_fragility.json").read_text())
share = r["score"]["share_removed"]
print(f"{'state':16} {'check':>6} {'kept':>6} {'damage':>7} {'share removed':>14}")
for k, v in r["reads"].items():
    s = share[k.removeprefix("revert_")] if k.startswith("revert_") else float("nan")
    print(f"{k:16} {v['check']:6.2f} {v['s']:6.2f} {v['damage']:7.3f} {s:14.2f}")

state             check   kept  damage  share removed
adjoint_lora       0.95   0.67   0.024            nan
move_s0_x0.5       0.99  -0.01   0.003            nan
move_s0_x1         1.03  -0.06   0.009            nan
move_s1_x1         1.07  -0.07   0.010            nan
nothing_lora       2.65   1.00   0.121            nan
revert_all         1.00   0.00   0.000           1.00
revert_attn        2.70   1.01   0.133          -0.42
revert_common      1.04   0.76   0.021           0.96
revert_layer0      1.35   0.86   0.168           0.70
revert_layer1      1.75   1.02   0.175           0.38
revert_layer2      1.90   1.01   0.128           0.25
revert_layer3      2.07   0.82   0.098           0.11
revert_mlp         1.00   0.47   0.025           1.00
revert_random      1.03   1.00   0.054           0.97
revert_rare        1.45   1.04   0.111           0.63
trained            2.20   1.00   0.234            nan
tt_lora            1.27   0.96   0.124            nan


## What we learned

On this broad fine-tune, the trained decomposition does not fail because its subcomponents moved this far: random
moves of the same weight size pass. The excess comes from the MLP subcomponents' changes acting together, partly
offset by the attention changes. Subcomponents Γ hardly uses carry less of it than a random half. A rank-8 LoRA fine-tune of the
same task gives a trained decomposition that passes. For `GOALS.md`, this points away from "the decomposition is too
brittle" and towards the update. A broad Adam update is hard for teacher training to place, and a low-rank one of the
same behaviour is not. The SGD half and the damage confound remain open.

## How compelling is the evidence

- **One adversary from one random start**, 20 steps: a lower bound. Every state here, passes included, could read
  higher under a longer search.
- **Noise floor.** The unchanged model's spread over three adversary seeds is 0.0748, a band from 1 to 1.075. E37's
  two batch orders differed by 14% in the check. Shares of excess removed less than 0.2 apart are not separated.
- **The random moves are matched in weight size only.** They do a twenty-fifth of the damage.
- **Reverts are not additive.** A share says what removing one group does with the rest in place, not what that group
  contributes alone.
- **Importance was read at the unchanged model on the check's rows**, not at the trained state or on other text.
- **One task, one batch order, one seed and one rate for the LoRA.** Gains were matched by held-out log-probability;
  damage was not matched.

## Surprises

- **A random move of the same size passes.** It was predicted to fail at 1.8 to 3, and read 1.03 and 1.07. Untested
  causes: the check may track output change, which random directions barely produce. Or the adversary may exploit the
  particular directions teacher training chose.
- **The MLP changes alone carry all the excess.** Reverting attention raised the check to 2.70. Untested cause: the
  attention changes may compensate for the MLP changes on Pile text, so removing them exposes the MLP excess.
- **The rarely important half removes less of the excess than a random half**, 0.63 against 0.97. Untested causes:
  that half may carry less of the weight change. Importance at the unchanged model may be the wrong ranking. Or any
  revert of half the change may break the combination the excess needs.
- **The LoRA's trained decomposition passes**, at 1.27. Untested causes: its low rank, its smaller change on plain
  text, or chance at one seed and one rate.
- **Layer 0's revert removed 0.70 of the excess**, a narrow miss of the 0.6 cap, inside the noise of a share.
- **Putting back a random half keeps the behaviour by mean log-probability and passes.** Its KL to the fine-tuned model
  is six times the trained decomposition's, so it is not a fix as it stands.

## What next

1. **The SGD amendment with smaller rates**, already planned. It answers the SGD half of the user's question.
2. **A random move matched in output change**, scaled so its damage on plain text equals the trained decomposition's.
   If it fails, the check punishes any change of this functional size. If it passes, the fragility is specific to
   where teacher training puts the change.
3. **The LoRA at a second seed, and at a rate matched on damage rather than gain.** A pass with damage matched points
   to rank. A fail points to the size of the change away from the task.

## Scorecard

Filed in `experiments/broad_fragility/results/predictions.json` before the run, unedited. Verdicts by the filed rules,
as `run.py`'s `score` wrote them.

| id | claim | read | verdict |
|---|---|---|---|
| P1 | a random move of the same size fails, mean of 2 seeds ≥ 1.61 | 1.049 | **miss**: passes, inside the band |
| P2 | the half-size move reads below the full-size mean | 0.988 against 1.049 | holds, within the noise |
| P3 | no single layer's revert removes more than 0.6 | layer 0: 0.705 | **miss**, narrow |
| P4 | rarely important half removes ≥ 0.2 more than a random half | 0.629 − 0.973 = −0.344 | **miss**, opposite way |
| P5 | SGD and LoRA stable rank below Adam's | LoRA 1.93 against 28.5; SGD no state | LoRA holds; not scored |
| P6 | SGD and LoRA trained decompositions fail, ≥ 1.61 | LoRA 1.273; SGD no state | LoRA **miss**; not scored |
| P7 | SGD and LoRA do-nothing fail, ≥ 1.61 | LoRA 2.653; SGD no state | LoRA holds; not scored |

## Not run

- The SGD arm. No rate in its grid learned, so none of its reads exist. An amendment with smaller rates is planned.
- A random move matched in damage or in output change; random moves along directions the inputs use.
- Reverts of single matrices, of each layer's MLP alone, or of the MLP's rarely important half alone.
- Importance read at the trained state, or on lowercase text.
- A second adversary seed or a longer adversary on any state; a second LoRA seed or rate; a second broad task.
- The LoRA's teacher-trained state is not on disk whole: both copies are truncated, so CPU follow-ups need a rerun.

**Provenance.** One RunPod H100 80GB HBM3, the job's log from 15:44 to 16:08. The plugin lost SSH to the pod right
after the job finished. The user copied the final results file and the log by hand. Results
`experiments/broad_fragility/results/broad_fragility.json`, status done; log `results/large/run.log`.